# NLP — rule-based parser and structured inference

This notebook inspects the implemented parser and baseline inference schema on existing master records. It displays evidence, uncertainty, and structured fields inline without running the writer-oriented CLI.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from liversira_ai.nlp.parser import parse_liver_report, split_sentences
from liversira_ai.nlp.baseline import extract_liver_lesion_features
MASTER_PATH = ROOT / 'data/processed/nlp/master_records.jsonl'
records = [json.loads(line) for line in MASTER_PATH.open(encoding='utf-8') if line.strip()]
print({'master_records': len(records), 'first_id': records[0]['id']})

## Parser examples and evidence

In [ ]:
examples = []
for record in records:
    text = record.get('reports', {}).get('narrative_clean')
    if text:
        parsed = parse_liver_report(text)
        if parsed.presence is not None:
            examples.append((record['id'], text, parsed))
    if len(examples) == 3:
        break
for record_id, text, parsed in examples:
    print(record_id)
    display(pd.DataFrame([{
        'sentences': split_sentences(text)[:3], 'presence': parsed.presence,
        'presence_reason': parsed.presence_reason, 'lesion_count': parsed.lesion_count,
        'location_segments': parsed.location_segments, 'dimensions_cm': parsed.reported_dimensions_cm,
        'max_diameter_cm': parsed.derived_max_diameter_cm, 'attenuation': parsed.attenuation,
    }]))
    display(pd.DataFrame([item.__dict__ for item in parsed.evidence]))

## Structured baseline output

In [ ]:
rows = []
for record in records[:200]:
    text = record.get('reports', {}).get('narrative_clean')
    features = extract_liver_lesion_features(text)
    rows.append({'id': record['id'], **{key: features[key] for key in ['has_lesion', 'lesion_count', 'location_segments', 'derived_max_diameter_cm', 'attenuation']}, 'unclear_fields': len(features['missing_or_unclear']), 'evidence_spans': len(features['evidence'])})
inference_table = pd.DataFrame(rows)
display(inference_table.head(10))
display(inference_table[['has_lesion', 'unclear_fields', 'evidence_spans']].value_counts().rename('records').reset_index())

## Existing JSONL schema and provenance inspection

In [ ]:
ground_truth_path = ROOT / 'data/processed/nlp/ground_truth.jsonl'
inference_path = ROOT / 'results/nlp/predictions/baseline_inferences_updated.jsonl'
gt_first = json.loads(next(ground_truth_path.open(encoding='utf-8')))
inf_first = json.loads(next(inference_path.open(encoding='utf-8')))
display(pd.DataFrame([{'ground_truth_keys': list(gt_first), 'inference_keys': list(inf_first), 'inference_input_field': inf_first['input_reference']['field'], 'inference_confidence': inf_first['confidence']}]))
print({'ground_truth_status': gt_first['validation']['status'], 'evidence_span_count': len(inf_first.get('evidence_spans', [])), 'missing_or_unclear_count': len(inf_first.get('missing_or_unclear', []))})

## Summary

The parser preserves uncertainty instead of treating absent evidence as a negative. Baseline records carry structured fields, evidence spans, and explicit missing/unclear fields. This notebook inspects those outputs and does not alter inference behavior or regenerate JSONL files.

In [ ]:
assert len(records) == 9262
assert inf_first['input_reference']['field'] == 'narrative_clean'
print('Parser and structured inference inspection completed without writing files.')